# Práctica Guiada 1 (Sesión 3): De Tabla Plana a Cubo Multidimensional OLAP (Low-Code)
### Asignatura: Minería de Datos (Posgrado TCOM - Universidad Anáhuac)

* **Estudiante:** [Nombre completo]
* **Matrícula / No. de Cuenta:** [Número de matrícula]
* **Fecha de Ejecución:** [Fecha]
* **Modalidad:** Taller Guiado Low-Code con Formularios Interactivos de Google Colab (`#@param`)

---
### 🎯 Objetivo General
Comprender la transición desde una **tabla de datos convencional (tabla plana o "sábana")** hacia una **estructura multidimensional (Esquema en Estrella y Cubo OLAP)**, ejecutando interactivamente las 4 operaciones analíticas fundamentales (*Roll-up, Drill-down, Slice & Dice y Pivot*) sobre métricas de comunicación digital sin necesidad de escribir código complejo.


## Paso 0: Generación e Importación del Dataset Plano ("Tabla Sábana")
En la práctica profesional de la comunicación digital, los datos suelen exportarse desde las plataformas (Google Ads, Meta Business, TikTok Ads) en **formatos planos convencionales**, donde cada fila repite constantemente información contextual (plataforma, ciudad, rango de edad) junto con métricas numéricas.

*Ejecuta la siguiente celda de código para generar automáticamente el dataset plano `df_plano` en tu sesión de Colab.*


In [3]:
import pandas as pd
import numpy as np

# Configurar semilla aleatoria para reproducibilidad
np.random.seed(42)
n_registros = 500

# Definir catálogos de dimensiones
fechas = pd.date_range(start='2026-01-01', periods=90, freq='D')
plataformas = ['Meta Ads', 'Google Ads', 'TikTok Ads']
edades = ['18-24', '25-34', '35-44', '45+']
regiones = ['CDMX', 'Monterrey', 'Guadalajara', 'Puebla']
contenidos = ['Video Corto (Reels/TikTok)', 'Imagen Estática', 'Anuncio de Texto (Search)', 'Carrusel Interactivo']

# Generación aleatoria del dataset plano
data_list = []
for i in range(n_registros):
    f = np.random.choice(fechas)
    p = np.random.choice(plataformas, p=[0.4, 0.35, 0.25])
    e = np.random.choice(edades, p=[0.35, 0.30, 0.20, 0.15])
    r = np.random.choice(regiones, p=[0.4, 0.25, 0.20, 0.15])
    c = np.random.choice(contenidos, p=[0.35, 0.25, 0.20, 0.20])
    
    imp = int(np.random.gamma(shape=2.0, scale=3000.0)) + 500
    clics = int(imp * np.random.uniform(0.02, 0.08)) + 5
    conv = int(clics * np.random.uniform(0.03, 0.12))
    costo = round(clics * np.random.uniform(8.0, 18.0), 2)
    
    fecha = pd.Timestamp(f)

    data_list.append({
        'id_evento': f"EVT-{10000 + i}",
        'fecha': fecha.strftime('%Y-%m-%d'),
        'mes': fecha.strftime('%B'),
        'trimestre': f"Q{(fecha.month - 1)//3 + 1}",
        'canal_plataforma': p,
        'rango_edad': e,
        'region_geografica': r,
        'tipo_contenido': c,
        'impresiones': imp,
        'clics': clics,
        'conversiones': conv,
        'costo_total': costo
    })

df_plano = pd.DataFrame(data_list)
print(f"✅ Dataset plano 'df_plano' generado con éxito: {df_plano.shape[0]} filas y {df_plano.shape[1]} columnas.")


✅ Dataset plano 'df_plano' generado con éxito: 500 filas y 12 columnas.


## Paso 1: Exploración Diagnóstica mediante Formulario Interactivo
Utiliza el formulario a continuación para ajustar cuántas filas deseas revisar y qué variable categórica deseas inspeccionar. Los formularios de Colab permiten interactuar con los datos modificando los controles sin escribir líneas de código adicionales.


In [4]:
#@title 🔍 Inspección Inicial del Dataset Plano
#@markdown Ajusta los controles del formulario y ejecuta la celda:
numero_filas = 5 #@param {type:"slider", min:1, max:20, step:1}
variable_a_analizar = "canal_plataforma" #@param ["canal_plataforma", "rango_edad", "region_geografica", "tipo_contenido", "mes"]

print(f"--- Muestra de las primeras {numero_filas} filas del dataset ---")
display(df_plano.head(numero_filas))

print(f"\n--- Conteo de registros por categoría ({variable_a_analizar.upper()}) ---")
display(df_plano[variable_a_analizar].value_counts().to_frame())


--- Muestra de las primeras 5 filas del dataset ---


,id_evento,fecha,mes,trimestre,canal_plataforma,rango_edad,region_geografica,tipo_contenido,impresiones,clics,conversiones,costo_total
0,EVT-10000,2026-02-21,February,Q1,TikTok Ads,35-44,Monterrey,Video Corto (Reels/TikTok),6660,146,17,2383.37
1,EVT-10001,2026-02-07,February,Q1,Meta Ads,45+,Monterrey,Anuncio de Texto (Search),10523,230,17,2759.68
2,EVT-10002,2026-03-01,March,Q1,Meta Ads,25-34,Guadalajara,Video Corto (Reels/TikTok),23406,1326,60,11470.58
3,EVT-10003,2026-01-04,January,Q1,TikTok Ads,25-34,CDMX,Video Corto (Reels/TikTok),7187,252,23,3553.19
4,EVT-10004,2026-01-08,January,Q1,Meta Ads,45+,CDMX,Anuncio de Texto (Search),6353,202,23,3181.77



--- Conteo de registros por categoría (CANAL_PLATAFORMA) ---


,count
canal_plataforma,
Meta Ads,202
Google Ads,170
TikTok Ads,128


## Paso 2: Conversión a Esquema en Estrella (*Star Schema*)
Para transformar la tabla plana en un **Almacén de Datos (Data Warehouse / Data Mart)** eficiente, separamos la información en dos componentes lógicos:
1. **Tablas de Dimensiones:** Listas maestras sin duplicados con los atributos descriptivos de contexto (*Tiempo, Plataforma, Audiencia, Contenido*).
2. **Tabla de Hechos:** Repositorio central que conserva las claves primarias (*Foreign Keys*) de cada dimensión y las métricas numéricas acumulables (*Impresiones, Clics, Conversiones, Costo*).

*Ejecuta el bloque a continuación para realizar la separación automática.*


In [5]:
#@title 🌟 Separación en Esquema en Estrella (Hechos y Dimensiones)
#@markdown Ejecuta esta celda para fragmentar la tabla plana en estructuras multidimensionales:

# 1. Creación de Tablas de Dimensiones
dim_plataforma = df_plano[['canal_plataforma']].drop_duplicates().reset_index(drop=True)
dim_plataforma['id_plataforma'] = [f"PL-{10 + i}" for i in range(len(dim_plataforma))]

dim_audiencia = df_plano[['rango_edad', 'region_geografica']].drop_duplicates().reset_index(drop=True)
dim_audiencia['id_audiencia'] = [f"AUD-{100 + i}" for i in range(len(dim_audiencia))]

dim_contenido = df_plano[['tipo_contenido']].drop_duplicates().reset_index(drop=True)
dim_contenido['id_contenido'] = [f"CONT-{10 + i}" for i in range(len(dim_contenido))]

dim_tiempo = df_plano[['fecha', 'mes', 'trimestre']].drop_duplicates().reset_index(drop=True)
dim_tiempo['id_tiempo'] = [f"TMP-{100 + i}" for i in range(len(dim_tiempo))]

# 2. Creación de la Tabla de Hechos (vinculando sólo IDs)
tabla_hechos = df_plano.merge(dim_plataforma, on='canal_plataforma') \
                       .merge(dim_audiencia, on=['rango_edad', 'region_geografica']) \
                       .merge(dim_contenido, on='tipo_contenido') \
                       .merge(dim_tiempo, on=['fecha', 'mes', 'trimestre'])

tabla_hechos = tabla_hechos[['id_evento', 'id_tiempo', 'id_plataforma', 'id_audiencia', 'id_contenido', 
                             'impresiones', 'clics', 'conversiones', 'costo_total']]

print("✅ Esquema en Estrella construido exitosamente.")
print(f"• Dim_Plataforma: {len(dim_plataforma)} registros")
print(f"• Dim_Audiencia: {len(dim_audiencia)} registros")
print(f"• Dim_Contenido: {len(dim_contenido)} registros")
print(f"• Dim_Tiempo: {len(dim_tiempo)} registros")
print(f"• Tabla de Hechos: {len(tabla_hechos)} eventos registrados.")

print("\n--- Muestra de la Tabla de Hechos (Primeras 5 filas) ---")
display(tabla_hechos.head())


✅ Esquema en Estrella construido exitosamente.
• Dim_Plataforma: 3 registros
• Dim_Audiencia: 16 registros
• Dim_Contenido: 4 registros
• Dim_Tiempo: 90 registros
• Tabla de Hechos: 500 eventos registrados.

--- Muestra de la Tabla de Hechos (Primeras 5 filas) ---


,id_evento,id_tiempo,id_plataforma,id_audiencia,id_contenido,impresiones,clics,conversiones,costo_total
0,EVT-10000,TMP-100,PL-10,AUD-100,CONT-10,6660,146,17,2383.37
1,EVT-10001,TMP-101,PL-11,AUD-101,CONT-11,10523,230,17,2759.68
2,EVT-10002,TMP-102,PL-11,AUD-102,CONT-10,23406,1326,60,11470.58
3,EVT-10003,TMP-103,PL-10,AUD-103,CONT-10,7187,252,23,3553.19
4,EVT-10004,TMP-104,PL-11,AUD-104,CONT-11,6353,202,23,3181.77


## Paso 3: Operación OLAP 1 — *Roll-up* (Subir en el Nivel de Agregación)
La operación **Roll-up** consiste en resumir o agrupar la información ascendiendo en una jerarquía conceptual (por ejemplo, ver los datos agrupados por **Mes** o por **Trimestre** en lugar de verlos día por día).


In [6]:
#@title 📈 Operación OLAP: Roll-up (Nivel de Agregación Temporal)
#@markdown Selecciona el nivel de resumen y la métrica que deseas agregar:
nivel_agregacion = "mes" #@param ["mes", "trimestre"]
metrica_a_sumar = "conversiones" #@param ["impresiones", "clics", "conversiones", "costo_total"]

rollup_resultado = df_plano.groupby(nivel_agregacion)[metrica_a_sumar].sum().reset_index()

print(f"--- Resultado del ROLL-UP por nivel: {nivel_agregacion.upper()} ---")
display(rollup_resultado)


--- Resultado del ROLL-UP por nivel: MES ---


,mes,conversiones
0,February,3234
1,January,3941
2,March,4892


## Paso 4: Operación OLAP 2 — *Drill-down* (Desglose a Mayor Nivel de Detalle)
La operación **Drill-down** es el proceso inverso al Roll-up. Consiste en descender en la jerarquía o añadir dimensiones secundarias para profundizar en el detalle de una plataforma o segmento específico.


In [7]:
#@title 🔍 Operación OLAP: Drill-down (Desglose por Región y Edad)
#@markdown Selecciona una plataforma para desglosar su comportamiento por audiencia y región:
plataforma_objetivo = "TikTok Ads" #@param ["Meta Ads", "Google Ads", "TikTok Ads"]
metrica_detalle = "costo_total" #@param ["impresiones", "clics", "conversiones", "costo_total"]

df_filtrado_p = df_plano[df_plano['canal_plataforma'] == plataforma_objetivo]
drilldown_resultado = df_filtrado_p.groupby(['region_geografica', 'rango_edad'])[metrica_detalle].sum().reset_index()

print(f"--- DRILL-DOWN para la plataforma: {plataforma_objetivo.upper()} ---")
display(drilldown_resultado.sort_values(by=metrica_detalle, ascending=False))


--- DRILL-DOWN para la plataforma: TIKTOK ADS ---


,region_geografica,rango_edad,costo_total
1,CDMX,25-34,92711.43
0,CDMX,18-24,91984.66
8,Monterrey,18-24,43062.48
4,Guadalajara,18-24,40126.52
12,Puebla,18-24,40110.65
11,Monterrey,45+,31760.15
5,Guadalajara,25-34,31223.40
10,Monterrey,35-44,25819.04
9,Monterrey,25-34,25251.24
3,CDMX,45+,24835.37


## Paso 5: Operación OLAP 3 — *Slice & Dice* (Extracción de Subcubos)
* **Slice (Rebanar):** Filtra una sola dimensión (p. ej., analizar únicamente la plataforma Meta Ads).
* **Dice (Subcubo):** Cruza filtros sobre dos o más dimensiones simultáneamente (p. ej., Meta Ads + Edad 18-24 + Formato Video Corto).


In [8]:
#@title 🍰 Operación OLAP: Slice & Dice (Subcubo de Audiencia)
#@markdown Configura los filtros para aislar un segmento específico de la campaña:
filtro_plataforma = "Meta Ads" #@param ["Todos", "Meta Ads", "Google Ads", "TikTok Ads"]
filtro_edad = "18-24" #@param ["Todos", "18-24", "25-34", "35-44", "45+"]
filtro_contenido = "Video Corto (Reels/TikTok)" #@param ["Todos", "Imagen Estática", "Video Corto (Reels/TikTok)", "Anuncio de Texto (Search)", "Carrusel Interactivo"]

sub_df = df_plano.copy()
if filtro_plataforma != "Todos":
    sub_df = sub_df[sub_df['canal_plataforma'] == filtro_plataforma]
if filtro_edad != "Todos":
    sub_df = sub_df[sub_df['rango_edad'] == filtro_edad]
if filtro_contenido != "Todos":
    sub_df = sub_df[sub_df['tipo_contenido'] == filtro_contenido]

totales = sub_df[['impresiones', 'clics', 'conversiones', 'costo_total']].sum()
cpa = (totales['costo_total'] / totales['conversiones']) if totales['conversiones'] > 0 else 0

print(f"--- SUBCUBO OBTENIDO ({len(sub_df)} eventos encontrados) ---")
print(f"💰 Inversión Total: ${totales['costo_total']:,.2f} MXN")
print(f"🎯 Conversiones Totales: {int(totales['conversiones']):,}")
print(f"📊 Costo Por Adquisición (CPA) Promedio: ${cpa:,.2f} MXN\n")
display(sub_df.head(10))


--- SUBCUBO OBTENIDO (28 eventos encontrados) ---
💰 Inversión Total: $98,893.92 MXN
🎯 Conversiones Totales: 644
📊 Costo Por Adquisición (CPA) Promedio: $153.56 MXN



,id_evento,fecha,mes,trimestre,canal_plataforma,rango_edad,region_geografica,tipo_contenido,impresiones,clics,conversiones,costo_total
44,EVT-10044,2026-01-17,January,Q1,Meta Ads,18-24,Guadalajara,Video Corto (Reels/TikTok),7993,205,24,2407.26
128,EVT-10128,2026-02-20,February,Q1,Meta Ads,18-24,Puebla,Video Corto (Reels/TikTok),12239,703,65,9558.33
150,EVT-10150,2026-01-26,January,Q1,Meta Ads,18-24,Monterrey,Video Corto (Reels/TikTok),765,54,5,713.47
192,EVT-10192,2026-03-29,March,Q1,Meta Ads,18-24,Guadalajara,Video Corto (Reels/TikTok),3592,206,17,1695.24
208,EVT-10208,2026-03-30,March,Q1,Meta Ads,18-24,Monterrey,Video Corto (Reels/TikTok),5204,138,8,2149.33
234,EVT-10234,2026-03-23,March,Q1,Meta Ads,18-24,CDMX,Video Corto (Reels/TikTok),12632,885,105,14337.11
266,EVT-10266,2026-01-13,January,Q1,Meta Ads,18-24,Monterrey,Video Corto (Reels/TikTok),1613,49,3,633.50
286,EVT-10286,2026-01-08,January,Q1,Meta Ads,18-24,Puebla,Video Corto (Reels/TikTok),5543,385,40,4358.94
295,EVT-10295,2026-03-09,March,Q1,Meta Ads,18-24,Guadalajara,Video Corto (Reels/TikTok),1624,117,9,2067.43
320,EVT-10320,2026-03-19,March,Q1,Meta Ads,18-24,CDMX,Video Corto (Reels/TikTok),2038,115,13,1396.95


## Paso 6: Operación OLAP 4 — *Pivot* (Matriz Cruzada Analítica)
La operación **Pivot (Rotación)** rota los ejes visuales del cubo de datos para enfrentar dos dimensiones en una matriz cruzada (filas vs. columnas) y calcular métricas agregadas o índices de costo por conversión (CPA).


In [9]:
#@title 🔄 Operación OLAP: Pivot Table (Matriz Cruzada)
#@markdown Selecciona las dimensiones para los ejes vertical y horizontal:
dim_filas = "canal_plataforma" #@param ["canal_plataforma", "rango_edad", "region_geografica"]
dim_columnas = "tipo_contenido" #@param ["tipo_contenido", "rango_edad", "region_geografica", "mes"]
valor_analizado = "Costo Por Adquisición (CPA)" #@param ["Conversiones Totales", "Costo Total", "Costo Por Adquisición (CPA)"]

if valor_analizado == "Conversiones Totales":
    matriz_pivot = pd.pivot_table(df_plano, values='conversiones', index=dim_filas, columns=dim_columnas, aggfunc='sum', fill_value=0)
elif valor_analizado == "Costo Total":
    matriz_pivot = pd.pivot_table(df_plano, values='costo_total', index=dim_filas, columns=dim_columnas, aggfunc='sum', fill_value=0)
else:
    costo_p = pd.pivot_table(df_plano, values='costo_total', index=dim_filas, columns=dim_columnas, aggfunc='sum', fill_value=0)
    conv_p = pd.pivot_table(df_plano, values='conversiones', index=dim_filas, columns=dim_columnas, aggfunc='sum', fill_value=0)
    matriz_pivot = (costo_p / conv_p.replace(0, np.nan)).round(2).fillna(0)

print(f"--- MATRIZ OLAP PIVOT: {dim_filas.upper()} vs {dim_columnas.upper()} ({valor_analizado}) ---")
display(matriz_pivot)


--- MATRIZ OLAP PIVOT: CANAL_PLATAFORMA vs TIPO_CONTENIDO (Costo Por Adquisición (CPA)) ---


tipo_contenido,Anuncio de Texto (Search),Carrusel Interactivo,Imagen Estática,Video Corto (Reels/TikTok)
canal_plataforma,,,,
Google Ads,174.78,203.14,158.29,158.63
Meta Ads,165.23,184.59,202.11,169.21
TikTok Ads,197.25,176.29,169.60,180.28


## Paso 7: Preguntas de Reflexión
Responde brevemente a las siguientes preguntas en una celda de texto o en tu reporte de entrega:

1. **¿Qué ventaja ofrece separar la "tabla plana" en un Esquema en Estrella con Tablas de Dimensiones y de Hechos?**
2. **Al comparar la operación *Roll-up* y *Drill-down*, ¿en qué caso práctico usarías cada una para tomar una decisión en una campaña publicitaria?**
3. **A partir de la matriz *Pivot* generada en el Paso 6, ¿cuál combinación de plataforma y tipo de contenido presentó el Costo Por Adquisición (CPA) más eficiente?**
